# Pandas for Augur - practice notebook

This is not a generic pandas tutorial. Every exercise here is an operation you will
actually use in Augur, on data shaped like Augur's real data.

**How to use it**

1. Run the setup cell once. It generates synthetic package data - no downloads needed.
2. Work top to bottom. Each section explains a concept, then gives you a task.
3. Write your answer in the cell marked `# YOUR CODE HERE`, then run it. The `check(...)`
   call will tell you if you got it right, and give a hint if not.
4. Solutions are at the very bottom. **Use them the way we agreed:** if you're stuck more
   than ~15 minutes on one exercise, look - then close it and rewrite from scratch without
   looking.

**Time:** about 4 hours. Sections 4, 7 and 10 are the ones that matter most for Augur.

## Part 0 - Setup

Run this once. It builds three DataFrames that mimic Augur's real tables:

| DataFrame | One row is | Mirrors |
|---|---|---|
| `packages` | one package | the 10,000-package universe |
| `snapshots` | one package in one week | the weekly snapshot table (Augur's core table) |
| `advisories` | one published advisory | the OSV bulk export - your label source |

The data is seeded, so your numbers will match mine exactly.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

rng = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# packages: the universe
# ---------------------------------------------------------------------------
PYPI = [
    "requests", "urllib3", "numpy", "pandas", "flask", "django", "pillow",
    "cryptography", "pyyaml", "jinja2", "sqlalchemy", "boto3", "click", "pytest",
    "setuptools", "certifi", "lxml", "scipy", "tornado", "celery", "werkzeug",
    "markupsafe", "idna", "chardet", "six", "python-dateutil", "pytz", "attrs",
    "packaging", "wheel", "virtualenv", "paramiko", "redis", "psycopg2", "pymongo",
    "aiohttp", "httpx", "starlette", "fastapi", "uvicorn", "pydantic", "typer",
    "rich", "tqdm", "matplotlib", "seaborn", "scikit-learn", "joblib", "protobuf",
    "grpcio", "google-auth", "passlib", "bcrypt", "sentry-sdk", "prometheus-client",
    "structlog", "gunicorn", "alembic", "marshmallow", "pyjwt",
]
NPM = [
    "lodash", "express", "react", "axios", "chalk", "minimist", "webpack", "moment",
    "debug", "commander", "async", "colors", "node-ipc", "ws", "socket.io", "mongoose",
    "ua-parser-js", "event-stream", "rc", "coa", "vue", "jquery", "underscore",
    "bluebird", "request", "node-fetch", "semver", "glob", "rimraf", "mkdirp",
    "yargs", "inquirer", "ora", "dotenv", "cors", "helmet", "body-parser", "morgan",
    "passport", "jsonwebtoken", "bcryptjs", "nodemon", "eslint", "prettier",
    "typescript", "rollup", "vite", "jest", "mocha", "chai", "sinon", "supertest",
    "cheerio", "puppeteer", "sharp", "multer", "ioredis", "pino", "winston", "nanoid",
]

names = PYPI + NPM
packages = pd.DataFrame({
    "package": names,
    "ecosystem": ["pypi"] * len(PYPI) + ["npm"] * len(NPM),
    "repo": ["gh/" + n for n in names],
})
# popularity is lognormal - a few giants, a long tail. Like the real world.
packages["base_downloads"] = np.exp(rng.normal(13.5, 1.9, len(packages))).round().astype(int)

# ---------------------------------------------------------------------------
# snapshots: one row per package per week, 2 years
# ---------------------------------------------------------------------------
WEEKS = 104
week_starts = pd.date_range("2023-01-02", periods=WEEKS, freq="7D")  # Mondays

rows = []
for _, p in packages.iterrows():
    trend = rng.normal(0.002, 0.004)       # slow growth or decline
    activity = rng.gamma(2.0, 3.0)         # how busy this repo is
    for i, w in enumerate(week_starts):
        downloads = p.base_downloads * (1 + trend) ** i * rng.normal(1.0, 0.08)
        rows.append({
            "package": p["package"],
            "week_start": w,
            "downloads": max(0, int(downloads)),
            "commits_7d": int(rng.poisson(activity)),
            "active_contributors": int(rng.poisson(max(0.5, activity / 2.5))),
            "open_issues": int(rng.poisson(activity * 4)),
            "releases_7d": int(rng.poisson(0.18)),
        })
snapshots = pd.DataFrame(rows)

# OpenSSF Scorecard does not cover every repo - so ~22% of scores are genuinely missing.
scorecard = rng.normal(6.0, 1.6, len(snapshots)).clip(0, 10).round(1)
scorecard[rng.random(len(snapshots)) < 0.22] = np.nan
snapshots["scorecard_score"] = scorecard

# ---------------------------------------------------------------------------
# advisories: the label source.
# Note: popular packages get MORE advisories (more eyes on them).
# That is the confounding you will have to beat with a popularity baseline.
# ---------------------------------------------------------------------------
# The exponent tempers the skew: popular packages attract more advisories, but not
# all of them - otherwise the bottom of the universe would have zero positives.
weights = packages["base_downloads"].astype(float) ** 0.30
weights = weights / weights.sum()
n_adv = 65
adv_pkgs = rng.choice(packages["package"], size=n_adv, p=weights)

published = pd.to_datetime("2023-01-02") + pd.to_timedelta(
    rng.integers(60, WEEKS * 7, size=n_adv), unit="D")

# In ~60% of real cases the fix ships BEFORE the advisory is published.
# That gap is the leakage Augur's blackout window exists to block.
fix_offset = np.where(rng.random(n_adv) < 0.60,
                      -rng.integers(1, 45, size=n_adv),   # fix lands first
                      rng.integers(0, 20, size=n_adv))    # advisory lands first
first_fix = published + pd.to_timedelta(fix_offset, unit="D")

advisories = pd.DataFrame({
    "advisory_id": ["OSV-2023-%04d" % i for i in range(1, n_adv + 1)],
    "package": adv_pkgs,
    "published_date": published,
    "first_fix_date": first_fix,
}).sort_values("published_date").reset_index(drop=True)

# a few advisories genuinely have no known fix yet
advisories.loc[rng.random(n_adv) < 0.08, "first_fix_date"] = pd.NaT

print("packages   ", packages.shape)
print("snapshots  ", snapshots.shape)
print("advisories ", advisories.shape)


# ---------------------------------------------------------------------------
# checker
# ---------------------------------------------------------------------------
def check(name, got, want, hint=""):
    """Compare your answer to the expected one."""
    if got is Ellipsis:
        print("[" + name + "] not answered yet - replace the ... with your code")
        return
    try:
        if isinstance(want, (pd.DataFrame, pd.Series)):
            ok = got.reset_index(drop=True).equals(want.reset_index(drop=True))
        elif isinstance(want, float):
            ok = abs(float(got) - want) < 1e-6
        else:
            ok = got == want
    except Exception as e:
        print("[" + name + "] could not compare: " + str(e))
        return
    if ok:
        print("[" + name + "] correct")
    else:
        print("[" + name + "] not yet.")
        print("   you returned: " + repr(got))
        if hint:
            print("   hint: " + hint)

## Part 1 - Look before you leap

Before any analysis, you look at the data. Every pandas session starts here.

| What | Tells you |
|---|---|
| `df.shape` | rows, columns |
| `df.head()` | what the values actually look like |
| `df.dtypes` | the types - **this is where bugs hide** |
| `df.info()` | types + non-null counts in one view |
| `df.describe()` | ranges and outliers for numeric columns |

The `dtypes` check matters more than it looks. A date column stored as a *string* compares
alphabetically, not chronologically - so `"2024-1-5" > "2024-12-31"` is `True`. That kind
of bug silently corrupts a whole label set.

In [ ]:
snapshots.head()

In [ ]:
snapshots.dtypes

**Exercise 1a.** How many rows are in `snapshots`? Assign the number to `n_rows`.

In [ ]:
# YOUR CODE HERE
n_rows = ...

check("1a", n_rows, len(packages) * WEEKS,
      "df.shape gives (rows, columns) - take the first element")

**Exercise 1b.** How many *distinct* packages appear in `advisories`?
Assign to `n_pkgs_with_advisories`.

This number matters in real life: it tells you how many packages can ever be a positive
example. If it is tiny, your positive class is tiny.

In [ ]:
# YOUR CODE HERE
n_pkgs_with_advisories = ...

check("1b", n_pkgs_with_advisories, advisories["package"].nunique(),
      "Series.nunique() counts distinct values")

## Part 2 - Filtering with boolean masks

A comparison on a Series gives you a Series of `True`/`False`. Pass that back into the
DataFrame and you keep only the `True` rows.

```python
mask = snapshots["downloads"] > 1_000_000
snapshots[mask]
```

Two rules that trip up everyone:

1. Use `&` and `|`, **not** `and` / `or`.
2. **Wrap each condition in parentheses.** `a > 1 & b < 2` parses wrong and raises.

```python
snapshots[(snapshots["downloads"] > 1_000_000) & (snapshots["commits_7d"] == 0)]
```

In [ ]:
mask = (snapshots["downloads"] > 1_000_000) & (snapshots["commits_7d"] == 0)
snapshots[mask].head()

**Exercise 2a.** Get every snapshot for the package `requests`. Assign to `req`.

In [ ]:
# YOUR CODE HERE
req = ...

check("2a", len(req), 104, "filter with snapshots['package'] == 'requests'")

**Exercise 2b.** Find snapshots that look like a **dormant but popular** package: more than
500,000 downloads, and zero commits in the week. Assign the *count* to `n_dormant`.

This is a real Augur signal - a heavily-used package nobody is maintaining.

In [ ]:
# YOUR CODE HERE
n_dormant = ...

_want = len(snapshots[(snapshots["downloads"] > 500_000) & (snapshots["commits_7d"] == 0)])
check("2b", n_dormant, _want, "two conditions, each in its own parentheses, joined by &")

## Part 3 - `value_counts`, and measuring a base rate

`value_counts()` counts how often each value appears. With `normalize=True` it gives
proportions instead of counts.

**This is how you will measure Augur's base rate in week 3** - the single number that
decides which metrics are meaningful. If only 2% of your snapshots are positive, then a
model that predicts "safe" for everything scores 98% accuracy while being worthless. That
is why your plan never mentions accuracy.

In [ ]:
advisories["package"].value_counts().head(10)

Look at that list. The packages at the top are the *popular* ones - and this is baked into
the synthetic data on purpose, because it is true in the real world too.

More users means more people auditing, which means more advisories found. That is the
**confounding** your popularity baseline (B0) exists to expose. The top of that list is
essentially what baseline **B1** (rank by prior advisory count) would predict.

**Exercise 3a.** What *proportion* of advisories belong to npm packages? Assign the float
to `npm_share`.

You will need `ecosystem`, which lives in `packages`, not `advisories` - so merge first.
(Part 6 covers merging; try it now and come back if you get stuck.)

In [ ]:
# YOUR CODE HERE
npm_share = ...

_want = float(advisories.merge(packages[["package", "ecosystem"]], on="package")
              ["ecosystem"].value_counts(normalize=True)["npm"])
check("3a", npm_share, _want, "merge to bring in ecosystem, then value_counts(normalize=True)")

## Part 4 - Dates and times

**This is the section that will bite you hardest in Augur.** Every label, every feature
window, every split depends on date arithmetic being right.

| Operation | Code |
|---|---|
| Parse strings to dates | `pd.to_datetime(s)` |
| Add or subtract time | `d + pd.Timedelta(days=90)` |
| Pull out a part | `s.dt.year`, `s.dt.month`, `s.dt.dayofweek` |
| Difference between dates | `(d2 - d1).dt.days` |
| Element-wise min of two columns | `df[["a", "b"]].min(axis=1)` |

That last one is not a detail - it is Augur's actual label rule.

In [ ]:
t = snapshots["week_start"].iloc[0]
print("snapshot date: ", t.date())
print("blackout ends: ", (t + pd.Timedelta(days=7)).date())
print("window ends:   ", (t + pd.Timedelta(days=90)).date())

**Exercise 4a - Augur's real label rule.**

An advisory's **event date** is the *earlier* of its published date and the date the first
fixed version shipped. Add a column `event_date` to `advisories` holding that value.

Careful: some rows have `NaT` (no known fix). `.min(axis=1)` skips missing values by
default, which is the behaviour you want here - if there is no fix date, the published date
stands alone.

In [ ]:
# YOUR CODE HERE
advisories["event_date"] = ...

_want = advisories[["published_date", "first_fix_date"]].min(axis=1)
check("4a", advisories["event_date"], _want,
      "df[['published_date','first_fix_date']].min(axis=1)")

**Exercise 4b - see the leakage for yourself.**

In how many advisories did the fix ship *before* the advisory was published? Assign the
count to `n_fix_first`.

Then read that number as a proportion of the total. **That** is why Augur anchors labels on the
event date and drops a 7-day blackout window. If you anchored on the publication date
instead, your model could learn to spot a fix already in progress and call it a prediction.
It would look brilliant on paper and be worthless in production.

In [ ]:
# YOUR CODE HERE
n_fix_first = ...

_want = int((advisories["first_fix_date"] < advisories["published_date"]).sum())
check("4b", n_fix_first, _want, "compare the two columns, then .sum() the boolean Series")

In [ ]:
print(str(n_fix_first) + " of " + str(len(advisories)) +
      " advisories were fixed before they were published")
print("that is {:.0%} of them".format(n_fix_first / len(advisories)))

## Part 5 - `groupby` and aggregation

Split the data into groups, compute something per group, put the results back together.
This is pandas' equivalent of SQL's `GROUP BY` - which you already know.

```python
snapshots.groupby("package")["downloads"].mean()
```

For several aggregations at once, named aggregation gives you clean column names:

```python
snapshots.groupby("package").agg(
    mean_downloads=("downloads", "mean"),
    total_commits=("commits_7d", "sum"),
)
```

In [ ]:
snapshots.groupby("package").agg(
    mean_downloads=("downloads", "mean"),
    total_commits=("commits_7d", "sum"),
    weeks_observed=("week_start", "count"),
).head()

**Exercise 5a.** Build a per-package summary called `pkg_stats` with exactly these columns,
in this order:

- `median_downloads` - median of `downloads`
- `total_commits` - sum of `commits_7d`
- `max_contributors` - max of `active_contributors`

Keep `package` as a regular column, not the index (use `.reset_index()`).

In [ ]:
# YOUR CODE HERE
pkg_stats = ...

_want = snapshots.groupby("package").agg(
    median_downloads=("downloads", "median"),
    total_commits=("commits_7d", "sum"),
    max_contributors=("active_contributors", "max"),
).reset_index()
check("5a", pkg_stats, _want, "use .agg() with named aggregation, then .reset_index()")

## Part 6 - `merge`: joining tables

Same idea as a SQL join.

```python
snapshots.merge(packages, on="package", how="left")
```

**The trap that will cost you a day if you do not know it.** If the right-hand table has
duplicate keys, a merge *multiplies* your rows. You think you joined; you actually
duplicated. And row counts in the hundreds of thousands make it easy to miss.

The defence is one argument:

```python
df.merge(other, on="package", how="left", validate="m:1")
```

`validate="m:1"` means "many rows on the left, at most one match on the right" - and pandas
**raises** if that is violated. Get in the habit of writing it on every merge. It is the
cheapest bug insurance in the library.

In [ ]:
enriched = snapshots.merge(packages[["package", "ecosystem"]],
                           on="package", how="left", validate="m:1")
print("before:", snapshots.shape)
print("after: ", enriched.shape)   # same row count - the merge did not explode
enriched.head(3)

**Exercise 6a.** Merge `advisories` with `packages` so each advisory carries its `ecosystem`
and `base_downloads`. Call the result `adv_enriched`.

Use a left join, and include `validate="m:1"`.

In [ ]:
# YOUR CODE HERE
adv_enriched = ...

check("6a", adv_enriched.shape[0], len(advisories),
      "a correct left join keeps exactly the original number of advisory rows")
print("columns:", list(adv_enriched.columns))

## Part 7 - Rolling windows (and the bridge to SQL window functions)

**Read this section twice.** It is the pandas twin of the SQL feature you have never used,
and between them they are how every Augur feature gets computed.

You know `GROUP BY`: it collapses many rows into one per group. A **window function** does
not collapse anything. It keeps every row and attaches an aggregate computed over a
*window* of neighbouring rows.

These two do the same job:

```sql
-- SQL
SELECT package, week_start, commits_7d,
       AVG(commits_7d) OVER (
         PARTITION BY package      -- separately for each package
         ORDER BY week_start       -- in date order
         ROWS BETWEEN 3 PRECEDING AND CURRENT ROW  -- this row + the 3 before it
       ) AS commits_4w_avg
FROM snapshots
```

```python
# pandas
snapshots.groupby("package")["commits_7d"].rolling(window=4).mean()
#         ^-- PARTITION BY         ^-- the window        ^-- the aggregate
```

The mapping is exact:

| SQL | pandas |
|---|---|
| `PARTITION BY package` | `.groupby("package")` |
| `ORDER BY week_start` | `.sort_values("week_start")` first |
| `ROWS BETWEEN 3 PRECEDING AND CURRENT ROW` | `.rolling(window=4)` |
| `AVG(...)` | `.mean()` |

**Sorting is not optional.** `rolling` walks rows in whatever order they are sitting in. If
the frame is not sorted by date within each package, you will silently average the wrong
weeks - and a feature built from the wrong weeks is a feature that may include the future.
That is leakage, and it will not announce itself.

In [ ]:
s = snapshots.sort_values(["package", "week_start"]).copy()

s["commits_4w_avg"] = (
    s.groupby("package")["commits_7d"]
     .rolling(window=4, min_periods=1)
     .mean()
     .reset_index(level=0, drop=True)   # drop the group level rolling adds back
)

s[s["package"] == "requests"][["week_start", "commits_7d", "commits_4w_avg"]].head(8)

Read those two columns side by side until the relationship is obvious. Row 4's average is
rows 1-4. Row 5's is rows 2-5. The window slides.

`min_periods=1` says "give me a value even when there are not 4 weeks of history yet."
Without it the first three rows would be `NaN`.

**Exercise 7a.** Add a column `downloads_8w_avg` to `s`: the 8-week rolling mean of
`downloads`, per package. Same pattern as above.

In [ ]:
# YOUR CODE HERE
s["downloads_8w_avg"] = ...

_want = (s.groupby("package")["downloads"].rolling(window=8, min_periods=1)
         .mean().reset_index(level=0, drop=True))
check("7a", s["downloads_8w_avg"].round(4), _want.round(4),
      "groupby -> rolling(window=8, min_periods=1) -> mean -> reset_index(level=0, drop=True)")

**Exercise 7b - a real Augur feature.** *Download slope* is one of the popularity features
in your plan: is this package growing or dying?

Add a column `downloads_pct_change_4w` to `s` holding each package's percentage change in
downloads versus 4 weeks earlier. Use `groupby(...).pct_change(periods=4)`.

(`pct_change` is itself a window function in disguise - it compares each row to one a fixed
distance behind it. In SQL that is
`LAG(downloads, 4) OVER (PARTITION BY package ORDER BY week_start)`.)

In [ ]:
# YOUR CODE HERE
s["downloads_pct_change_4w"] = ...

_want = s.groupby("package")["downloads"].pct_change(periods=4)
check("7b", s["downloads_pct_change_4w"].round(6), _want.round(6),
      "s.groupby('package')['downloads'].pct_change(periods=4)")

## Part 8 - Missing data, and when *not* to fill it

`scorecard_score` is missing for about 22% of rows - because OpenSSF Scorecard genuinely
does not cover every repository. That is not a data error, it is a fact about the world.

| Operation | Code |
|---|---|
| Count missing per column | `df.isna().sum()` |
| Drop rows with missing | `df.dropna(subset=["col"])` |
| Fill | `df["col"].fillna(value)` |

**The judgement call that matters here.** The reflex is to fill missing values with the
mean. Resist it. "This repo has no Scorecard entry" may itself be a risk signal - filling it
with the average erases that information and tells the model a comfortable lie.

LightGBM handles `NaN` natively: it learns which branch missing values should take. So for
Augur, leaving the gap is usually the better choice, and *"why didn't you impute?"* is a
question you will now be able to answer.

In [ ]:
snapshots.isna().sum()

**Exercise 8a.** What fraction of `scorecard_score` values are missing? Assign the float to
`missing_frac`.

In [ ]:
# YOUR CODE HERE
missing_frac = ...

_want = float(snapshots["scorecard_score"].isna().mean())
check("8a", missing_frac, _want,
      "isna() gives booleans; .mean() of booleans is the proportion")

## Part 9 - Vectorize; do not loop

pandas operations run as compiled array operations underneath. Writing a Python loop, or
`.apply()` with a lambda, throws that away and runs your code one row at a time.

On 60 packages it is invisible. On Augur's real table - 10,000 packages times 100+ weekly
snapshots, so over a million rows - it is the difference between a second and several
minutes, on something you will rerun constantly.

In [ ]:
import time

big = pd.concat([snapshots] * 4, ignore_index=True)   # ~50k rows

t0 = time.perf_counter()
_ = big.apply(lambda r: r["downloads"] / max(r["commits_7d"], 1), axis=1)
t_apply = time.perf_counter() - t0

t0 = time.perf_counter()
_ = big["downloads"] / big["commits_7d"].clip(lower=1)
t_vec = time.perf_counter() - t0

print("apply(axis=1): {:.3f}s".format(t_apply))
print("vectorized:    {:.4f}s".format(t_vec))
print("speedup:       {:.0f}x".format(t_apply / max(t_vec, 1e-9)))

The rule of thumb: **if you are writing `.apply(..., axis=1)`, there is almost always a
vectorized form.** Reach for column arithmetic, `np.where`, `.clip()`, `.map()` instead.

## Part 10 - Capstone: label the snapshots

Everything so far was preparation for this. You are about to build Augur's label rule - in
miniature, on fake data, but it is the same logic you will write for real in **week 3**.

**The rule.** For a snapshot of package `p` at week `t`, look at every advisory event date
for `p`:

| Condition | Label |
|---|---|
| an event falls in `(t, t + 7 days]` | **drop the row** (return `None`) |
| an event falls in `(t + 7 days, t + 90 days]` | `1` |
| otherwise | `0` |

Check the blackout **first**. A row with an event 3 days out is dropped even if another
event also falls at day 40 - the blackout wins, because that row is contaminated.

Why drop instead of labelling negative? Because those rows are neither. The fix is already
underway, so the signals are polluted by the response to the vulnerability. Calling them
negative would teach the model something false; keeping them as positive is the leakage
itself. Dropping is the only honest option.

In [ ]:
# a lookup from package -> sorted array of its event dates
events_by_pkg = (
    advisories.dropna(subset=["event_date"])
              .groupby("package")["event_date"]
              .apply(lambda x: np.sort(x.values))
              .to_dict()
)
print("packages with at least one event:", len(events_by_pkg))
_first = list(events_by_pkg)[0]
print("example:", _first, "->", events_by_pkg[_first][:3])

**Exercise 10a.** Complete `label_snapshot`. Return `1`, `0`, or `None`.

In [ ]:
def label_snapshot(package, t, events_by_pkg):
    """Label one snapshot.

    Returns 1 (advisory in the prediction window), 0 (none), or None (drop the row).
    """
    events = events_by_pkg.get(package)
    if events is None or len(events) == 0:
        return 0

    blackout_end = t + pd.Timedelta(days=7)
    window_end = t + pd.Timedelta(days=90)

    # comparing numpy datetime64 to pandas Timestamp works, but being explicit is safer:
    t64 = np.datetime64(t)
    b64 = np.datetime64(blackout_end)
    w64 = np.datetime64(window_end)

    # TODO 1: if any event is in (t64, b64]      -> return None
    # TODO 2: if any event is in (b64, w64]      -> return 1
    # TODO 3: otherwise                          -> return 0
    # (np.any((events > a) & (events <= b)) tests "is there an event in (a, b]")

    # YOUR CODE HERE
    raise NotImplementedError


# sanity checks: one package with a single event on 2023-03-01
_ev = {"x": np.array([np.datetime64("2023-03-01")])}
_cases = [
    ("2023-01-20", 1,    "event 40 days ahead  -> inside the window"),
    ("2023-02-26", None, "event 3 days ahead   -> inside the blackout, drop"),
    ("2022-10-01", 0,    "event 151 days ahead -> beyond the 90-day window"),
    ("2023-06-01", 0,    "event already in the past"),
]
for _d, _expect, _why in _cases:
    _got = label_snapshot("x", pd.Timestamp(_d), _ev)
    _mark = "ok " if _got == _expect else "BAD"
    print("%s %s  got=%-5s expect=%-5s  %s" % (_mark, _d, _got, _expect, _why))

**Exercise 10b.** Apply it to every snapshot, then measure the base rate.

In [ ]:
s["label"] = [label_snapshot(p, t, events_by_pkg)
              for p, t in zip(s["package"], s["week_start"])]

dropped = s["label"].isna().sum()
labeled = s.dropna(subset=["label"]).copy()
labeled["label"] = labeled["label"].astype(int)

print("total snapshots:    {:,}".format(len(s)))
print("dropped (blackout): {:,}".format(dropped))
print("labeled:            {:,}".format(len(labeled)))
print()
print("BASE RATE")
print(labeled["label"].value_counts(normalize=True).round(4))

**Stop and look at that base rate.** That single number decides everything downstream:

- It is why **accuracy is meaningless** - predicting all-zeros scores whatever the negative
  share is, while catching nothing.
- It is why **PR-AUC beats ROC-AUC** as your headline metric. ROC-AUC stays flattering at a
  low base rate because the enormous negative class keeps the false-positive *rate* tiny.
  PR-AUC does not let you off.
- It is why your plan says *"measure the base rate in week 3 and write it down."*

**Exercise 10c - see the confounding with your own eyes.**

Split the labeled snapshots into the top 25% by downloads and the bottom 25%, and compare
the positive rate in each.

In [ ]:
q75 = labeled["downloads"].quantile(0.75)
q25 = labeled["downloads"].quantile(0.25)

top_rate = labeled[labeled["downloads"] >= q75]["label"].mean()
bottom_rate = labeled[labeled["downloads"] <= q25]["label"].mean()

print("positive rate, top 25% by downloads:    {:.2%}".format(top_rate))
print("positive rate, bottom 25% by downloads: {:.2%}".format(bottom_rate))
print("ratio: {:.1f}x".format(top_rate / max(bottom_rate, 1e-9)))

**That gap is baseline B0.** Sorting by downloads alone - no model, no features, no machine
learning - already separates risky from safe packages.

Which is exactly why your plan insists the model must beat it. A LightGBM that scores well
but does not beat this is not detecting risk. It is detecting popularity with extra steps,
and an interviewer who asks *"did you compare against a popularity baseline?"* would end
that conversation in one question.

You now have the intuition for the most important argument in the whole project.

## What to do next

1. If any section felt shaky, redo it tomorrow **without** looking at your earlier answer.
2. Section 7 is the one to be fluent in - it is both your pandas rolling windows and your
   SQL window functions, and it is how every Augur feature gets built.
3. Then tell me you are through, and we start week 1: the OSV bulk export and the
   package-to-repo mapping.

Concepts you have now met in code rather than in the abstract, all of which you will be
asked about: base rate, confounding, leakage, the blackout window, and why PR-AUC is the
headline metric.

---

# Solutions

**Do not read these until you have tried.** And when you do read one: close it, then
rewrite the answer from scratch without looking. Reading a solution feels like learning and
is not.

```python
# 1a
n_rows = snapshots.shape[0]

# 1b
n_pkgs_with_advisories = advisories["package"].nunique()

# 2a
req = snapshots[snapshots["package"] == "requests"]

# 2b
n_dormant = len(snapshots[(snapshots["downloads"] > 500_000)
                          & (snapshots["commits_7d"] == 0)])

# 3a
_m = advisories.merge(packages[["package", "ecosystem"]],
                      on="package", how="left", validate="m:1")
npm_share = float(_m["ecosystem"].value_counts(normalize=True)["npm"])

# 4a
advisories["event_date"] = advisories[["published_date", "first_fix_date"]].min(axis=1)

# 4b
n_fix_first = int((advisories["first_fix_date"] < advisories["published_date"]).sum())

# 5a
pkg_stats = snapshots.groupby("package").agg(
    median_downloads=("downloads", "median"),
    total_commits=("commits_7d", "sum"),
    max_contributors=("active_contributors", "max"),
).reset_index()

# 6a
adv_enriched = advisories.merge(
    packages[["package", "ecosystem", "base_downloads"]],
    on="package", how="left", validate="m:1")

# 7a
s["downloads_8w_avg"] = (
    s.groupby("package")["downloads"]
     .rolling(window=8, min_periods=1).mean()
     .reset_index(level=0, drop=True))

# 7b
s["downloads_pct_change_4w"] = s.groupby("package")["downloads"].pct_change(periods=4)

# 8a
missing_frac = float(snapshots["scorecard_score"].isna().mean())

# 10a  (the body, after the two Timedelta lines)
    if np.any((events > t64) & (events <= b64)):
        return None                      # blackout checked FIRST
    if np.any((events > b64) & (events <= w64)):
        return 1
    return 0
```